AI agents how to interact with them using llm

In [1]:
from google import genai
from google.genai import types
import os
import requests

In [2]:
client = genai.Client(api_key=os.getenv("API_KEY"))

In [3]:
def getSum(num1, num2):
    return num1+num2+10

In [4]:
def getPrime(num):
    return True

In [5]:
def getCountry(country):
    url = f"https://restcountries.com/v3.1/name/{country}"

    response = requests.get(url)
    if response.status_code!=200:
        return "Could not get the details of country"
    country_details = {
    item['tld'][0]: {  # Using the first item of the tld list as the key
        "cca2": item.get('cca2'),
        "independent": item.get('independent'),
        "unMember": item.get('unMember'),
        "capital": item.get('capital'),
        "altSpellings": item.get('altSpellings'),
        "region": item.get('region'),
        "borders": item.get('borders', []), # Default to empty list if no borders
        "area": item.get('area'),
        "continents": item.get('continents'),
        "nativeName": item.get('name', {}).get('nativeName')
    }
        for item in response.json()
    }
    return country_details

defining func declaration used by model

In [6]:

get_sum_function = {
    "name": "getSum",
    "description": "Returns sum of given two numbers",
    "parameters":{
        "type": "object",
        "properties":{
            "num1":{
                "type": "integer",
                "description": "First number for addition ex: 10"
            },
             "num2":{
                "type": "integer",
                "description": "Second number for addition ex: 12"
            }
        },
        "required": ["num1","num2"]
    }
}

In [7]:
get_prime_function = {
    "name": "getPrime",
    "description": "Returns if given number is prime no or not ex: True",
    "parameters": {
        "type": "object",
        "properties": {
            "num":{
                "type": "integer",
                "description": "Number which needs to be checked if it is prime"
            }
        },
        "required":["num"]
    }
}

In [9]:
get_country_function = {
    "name": "getCountry",
    "description": "Returns dictionary of details related to country like cca2,independent,unMember,capital,altSpellings,region,borders,area,continents,nativeNames",
    "parameters": {
        "type": "object",
        "properties": {
            "country":{
                "type": "string",
                "description": "Country name to be searched eg. india"
            }
        },
        "required":["country"]
    }
}

Configuring created tools(Functions) with LLM client

In [24]:
tools = types.Tool(function_declarations=[get_sum_function,
            get_prime_function,get_country_function])
tools_map = {'getSum': getSum, 'getPrime': getPrime,'getCountry':getCountry}
config = types.GenerateContentConfig(tools=[tools])

In [27]:
chat = client.chats.create(
    model="gemini-2.5-flash",
    config=config,
)


In [28]:
user_input = None
while True:
    if not user_input:
        user_input = input("Enter your message")
        print(user_input)
    response = chat.send_message(user_input)
    if response.function_calls:
        function_call = response.function_calls[0]
        function_name = function_call.name
        function_args = function_call.args
        if function_name in tools_map:
            function_result = tools_map[function_name](**function_args)
            response = chat.send_message(
                types.Part.from_function_response(
                    name = function_name,
                    response={"result": function_result}
                )
            )
    else:
        user_input = None
        print(f"AI: {response.text}")
        


sum of 4 and 4


ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 5, model: gemini-2.5-flash\nPlease retry in 4.453832834s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerMinutePerProjectPerModel-FreeTier', 'quotaDimensions': {'model': 'gemini-2.5-flash', 'location': 'global'}